# Failure-start selection: one new arm
A simple fixed-pool adaptation inspired by [Mulligan](https://arxiv.org/html/2610.05882v1): retrieve demonstrations near failed initial states, then fill the remaining acquisition slots with initial-state diversity. **Not a reproduction:** the paper collects new supervision at selected starts; this notebook retrieves existing demonstrations at nearby starts. No classifier, actor/critic changes, or human correction collection.

Start with **one round** from the existing 20k baseline: 20 diagnostic rollouts, 20 added expert episodes, 10k policy updates, and 200 evaluation rollouts. Existing arms are read, not retrained. DDPM and execute=8 remain inherited from the original protocol. Optional extension to three rounds uses fresh parent-policy diagnostics each round.

New directory: `failure_start_v1_seed11`. Preserve prior results, including negative findings. This is exploratory: prior final results informed the redesign, and deciding whether to extend after seeing round-one results adds further selection bias. Confirm a fixed method on fresh starts and more training seeds later.


In [ ]:
from pathlib import Path
import subprocess, sys
REPOSITORY = "https://github.com/0-5-blood-prince/robotics-data-flywheel.git"
REVISION = "main"  # Set to a commit SHA to reproduce a particular code version.
ROOT = Path('/content/robotics-data-flywheel')
if any(name.startswith('robot_flywheel') for name in sys.modules):
    raise RuntimeError('Restart the runtime before refreshing project code. Saved Drive checkpoints remain available.')
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(ROOT)], check=True)
else:
    if not (ROOT / '.git').exists():
        raise RuntimeError('This directory came from an old ZIP. Start a fresh Colab runtime once to switch to GitHub setup.')
    changes = subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'],text=True)
    if changes.strip(): raise RuntimeError('Local code changes found; preserve them before refreshing.')
subprocess.run(['git','-C',str(ROOT),'fetch','origin',REVISION],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
CODE_COMMIT = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('Code commit:', CODE_COMMIT)
import os, importlib
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{ROOT}[simulation]'], check=True)
# Activate src in this already-running kernel: editable-install .pth files
# are normally processed only when a Python interpreter starts.
sys.path.insert(0, str(ROOT / 'src'))
importlib.invalidate_caches()
import robot_flywheel
assert Path(robot_flywheel.__file__).resolve().is_relative_to(ROOT.resolve())
print('Package ready:', robot_flywheel.__file__)

# Check the native simulator API before dataset preparation.
subprocess.run([sys.executable, '-c',
    "import gymnasium as gym, gym_pusht; e=gym.make('gym_pusht/PushT-v0'); e.reset(seed=0); e.step(e.action_space.sample()); e.close(); print('Simulator ready')"], check=True)


## Persistence and runtime
Set `USE_DRIVE=True` to preserve checkpoints across runtime replacement. Checkpoints are saved every 1,000 updates; rerunning training resumes automatically. Only load checkpoints produced by this project or another trusted source.

In [ ]:
import os, sys, json, torch, numpy as np
os.environ['SDL_VIDEODRIVER'] = 'dummy'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/robotics-data-flywheel-runs')
else:
    WORK = ROOT / 'runs'
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'code_commit.txt').write_text(CODE_COMMIT)
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.cuda.is_available(),
       'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})
if not torch.cuda.is_available():
    print('Switch to a GPU runtime before substantial training. CPU is suitable for tests only.')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Load data and verify simulator compatibility

In [ ]:
from robot_flywheel.prepare import prepare
from robot_flywheel.data import load_zarr
EPISODES, SOURCE = prepare(ROOT / 'data/pusht')
display(SOURCE)


In [ ]:
import matplotlib.pyplot as plt
from robot_flywheel.rollout import check_replay
COMPATIBILITY = check_replay(EPISODES, WORK / 'compatibility', count=3)
display(COMPATIBILITY)
fig, axs = plt.subplots(1,3,figsize=(15,4))
for ax, episode in zip(axs, EPISODES[:3]):
    d = np.load(WORK / 'compatibility' / f'replay_{episode.id}.npz')
    ax.plot(d['recorded'][:,2],d['recorded'][:,3],label='Recorded object')
    ax.plot(d['simulated'][:,2],d['simulated'][:,3],label='Replayed object')
    ax.set(title=f'Episode {episode.id}',aspect='equal'); ax.legend()
plt.show()
assert COMPATIBILITY['passed'], 'Compatibility failed. Inspect report/traces before training.'


## 1. Choose the new arm budget
Default `ROUNDS_TO_RUN=1`. Set to 3 later in the **same unchanged run** to extend; completed training and diagnostic rollouts are reused. Keep code, data, and selector settings fixed for resume.

Half of 20 acquisition slots are reserved for failure matching. Failed rollouts are ordered by increasing maximum coverage (severity proxy), then seed. For each, pick the nearest unacquired expert initial state. If fewer than 10 failures exist, unused slots go to diversity. Every demonstration is selected at most once.

Distances use object XY / 512, agent-relative XY / 1024, and sin/cos orientation / 2. Coverage fill greedily maximizes distance to the original and already selected initial states. Large match distances mean a weak proxy for the requested start; they are logged, not silently treated as exact reset supervision. These are fixed heuristic choices, not tuned claims of optimality.


In [ ]:
from robot_flywheel.initial_state import StartConfig, run_failure_start, compare_failure_start
ORIGINAL = WORK / 'exploratory_ddpm_v1_seed11'
GUIDED = WORK / 'success_guided_v3_seed11'  # Optional existing comparison only.
OUTPUT = WORK / 'failure_start_v1_seed11'
CONFIG = StartConfig(failure_fraction=0.5)
ROUNDS_TO_RUN = 1
assert (ORIGINAL/'frozen_protocol.json').exists(), f'Missing original study: {ORIGINAL}'
print('New experiment:',OUTPUT)
print('Rounds requested:',ROUNDS_TO_RUN)


## 2. Collect diagnostic failures, select demonstrations, and train only the new arm
Final evaluation data never enters selection. Every round starts from its own parent policy; round one inherits the original baseline including optimizer, EMA and normalization. Interruptions can be resumed by rerunning unchanged.

In [ ]:
CHECKPOINTS = run_failure_start(
    EPISODES, SOURCE, ORIGINAL, OUTPUT, COMPATIBILITY,
    CONFIG, rounds_to_run=ROUNDS_TO_RUN, final_evaluation=False)
display(CHECKPOINTS)


## 3. Inspect acquisition explanations
Points show object starting positions; lines connect failed starts to retrieved expert starts. Distance also includes agent-relative position and periodic angle, so XY closeness alone does not determine the match. Coverage-fill points target unexplored starting configurations, not failure states.

In [ ]:
import pandas as pd
SEED = json.loads((ORIGINAL/'frozen_protocol.json').read_text())['protocol']['training_seeds'][0]
by_id = {ep.id:ep for ep in EPISODES}
for r in range(1,ROUNDS_TO_RUN+1):
    selection=json.loads((OUTPUT/f'seed_{SEED}/failure_start/round_{r}/selection.json').read_text())
    display(pd.DataFrame(selection['explanations']))
    fig,ax=plt.subplots(figsize=(7,6))
    for row in selection['explanations']:
        state=by_id[row['episode_id']].states[0]
        if row['reason']=='failure_start_nearest':
            failed=np.asarray(row['failure_initial_state'])
            ax.plot([failed[2],state[2]],[failed[3],state[3]],color='gray',alpha=.5)
            ax.scatter(failed[2],failed[3],marker='x',color='red')
            ax.scatter(state[2],state[3],color='blue')
        else:ax.scatter(state[2],state[3],color='green',marker='s')
    ax.set(xlim=(0,512),ylim=(0,512),aspect='equal',title=f'Round {r}: red failures, blue matches, green coverage fill')
    plt.show()
    matches=[x['match_distance'] for x in selection['explanations'] if 'match_distance' in x]
    print({'failed_diagnostics':selection['failures'],'matched_episodes':len(matches),
           'mean_match_distance':float(np.mean(matches)) if matches else None,
           'max_match_distance':max(matches) if matches else None})


## 4. Evaluate new checkpoints only
200 paired starts per new checkpoint; completed evaluations are cached. Existing arms are not rerun. The stage labels printed first reflect resume checks, not retraining.

In [ ]:
RESULTS = run_failure_start(
    EPISODES, SOURCE, ORIGINAL, OUTPUT, COMPATIBILITY,
    CONFIG, rounds_to_run=ROUNDS_TO_RUN, final_evaluation=True)
display(RESULTS)


## 5. Compare with matching rounds
A one-round run compares against **round one** of random, diversity, continued training, original targeting, and optionally success-guided selection. The chart retains later existing results for context; those are not matched-budget comparators for a one-round result. Intervals are conditional on one training seed.

In [ ]:
COMPARISONS, figure = compare_failure_start(
    ORIGINAL, OUTPUT, guided=GUIDED if (GUIDED/'results.json').exists() else None)
plt.show();display(COMPARISONS)
display(pd.read_csv(OUTPUT/'combined_results.csv'))


## 6. Saved paired videos and artifacts
Shows the first evaluation seed for the new method and matching-round random control. This is a predetermined example, not a selected success. No extra rollouts are generated.

In [ ]:
from IPython.display import Video, FileLink
start=json.loads((ORIGINAL/'frozen_protocol.json').read_text())['protocol']['evaluation_start']
for arm,home in [('random',ORIGINAL),('failure_start',OUTPUT)]:
    path=home/f'seed_{SEED}'/arm/f'round_{ROUNDS_TO_RUN}'/'final_evaluation'/f'episode_{start}.mp4'
    print(arm)
    if path.exists():display(Video(str(path),embed=True,width=360))
    else:print('No saved video for this start.')
for name in ['selector_contract.json','results.json','combined_results.csv','comparisons.json','combined_curves.png']:
    display(FileLink(str(OUTPUT/name)))
print('To extend: set ROUNDS_TO_RUN=3 and rerun sections 2–6. Do not change CONFIG or OUTPUT.')
